# Day 24 — Lecture 1: Feature Engineering - Encoding

> Source: Apna College Prime AI/ML • Day 24 Lecture 1 • Video Duration: `07:56`
> Core Objective: Categorical columns (`sex`, `smoker`, `region`) ko numeric values me convert karna using binary mapping aur `pd.get_dummies()`.
> Key Parameters: `columns=["region"]`, `drop_first=False`, `dtype=int`
> Dataset Used: `insurance.csv`
> Core Concept: Machine Learning models sirf numbers par kaam karte hain — One-Hot Encoding har category ke liye ek naya 0/1 binary column banata hai

## 🛠️ Step 0: Setup & Imports

Chalo zaroori libraries import karte hain aur `insurance.csv` ko `insurance_data` DataFrame me load karte hain.
- **Target:** Medical `charges` predict karna based on age, sex, bmi, children, smoker aur region.

In [ ]:
# 1. Libraries import karo
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 2. Insurance dataset load karo
insurance_data = pd.read_csv("insurance.csv")

# 3. Dataset ki shape aur top rows dekho
print("Shape:", insurance_data.shape)
print(insurance_data.head(3))

## 📍 Step 1: Day 23 Baseline Recap (`region` Dropped)

Dekho Day 23 me humne sirf `sex` aur `smoker` ko `map()` kiya tha aur `region` column ko drop kar diya tha (`R² = 0.78113`).

| Column | Categories | Mapping Rule |
|---|---|---|
| `sex` | `female`, `male` | `female -> 1`, `male -> 0` |
| `smoker` | `yes`, `no` | `yes -> 1`, `no -> 0` |

In [ ]:
# 1. Baseline ke liye charges aur region dono drop karo
X_base = insurance_data.drop(columns=["charges", "region"])
y_base = insurance_data["charges"]

# 2. Binary mapping apply karo
X_base["sex"] = X_base["sex"].map({"female": 1, "male": 0})
X_base["smoker"] = X_base["smoker"].map({"yes": 1, "no": 0})

# 3. Encoded columns verify karo
print("Baseline features:", list(X_base.columns))
print(X_base.head(2))

## 🎯 Step 2: Baseline `LinearRegression` Score Check

Chalo `region` ke bina baseline `LinearRegression()` run karke benchmark $R^2$ score confirm karte hain.

In [ ]:
# 1. 80-20 ratio me train-test split karo
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42
)

# 2. Model train karo aur test predictions lo
base_model = LinearRegression()
base_model.fit(X_train_b, y_train_b)
y_pred_b = base_model.predict(X_test_b)

# 3. Baseline R-squared print karo
print("Baseline r-squared:", r2_score(y_test_b, y_pred_b))

## ⚙️ Step 3: One-Hot Encoding with `pd.get_dummies()`

Ab `region` ko drop nahi karenge — isme 4 unique categories hain (`northeast`, `northwest`, `southeast`, `southwest`).
- **One-Hot Encoding:** `pd.get_dummies()` har category ke liye alag dummy column banata hai.
- **Default Output:** Bina `dtype=int` ke Pandas `True`/`False` boolean columns deta hai.

In [ ]:
# 1. Ab sirf charges drop karo taaki region features me rahe
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]

# 2. Region par get_dummies lagao (boolean output check)
X_bool = pd.get_dummies(X, columns=["region"], drop_first=False)

# 3. Naye boolean dummy columns dekho
region_cols = [c for c in X_bool.columns if "region_" in c]
print("Dummy columns:", region_cols)
print(X_bool[region_cols].head(3))

## 🔬 Step 4: Clean Integer Conversion (`dtype=int`) & Mapping

Yaad rakhna: Models ke liye `dtype=int` pass karne se `True`/`False` seedha `1`/`0` integers me convert ho jate hain, saath me `sex` aur `smoker` bhi map karenge.

In [ ]:
# 1. dtype=int pass karke 0/1 integer dummy columns banao
X = pd.get_dummies(X, columns=["region"], drop_first=False, dtype=int)

# 2. Sex aur smoker columns ko 1/0 me map karo
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})

# 3. Complete numeric feature matrix verify karo
print("Total columns:", X.shape[1])
print(X.head(3))

## 📈 Step 5: Model Retrain & $R^2$ Comparison

Ab encoded features ke saath dobara `LinearRegression()` fit karke dekhte hain ki accuracy par kya effect aaya.

| Stage | Region Handling | Test $R^2$ Score |
|---|---|---|
| Day 23 Baseline | `region` dropped | `0.78113` |
| Day 24 One-Hot | 4 dummy columns added | `0.78359` |

In [ ]:
# 1. Encoded features par train-test split karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. LinearRegression fit aur predict karo
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 3. Improved R-squared score print karo
r2 = r2_score(y_test, y_pred)
print("r2 score:", r2)

## 🏆 Step 6: Next Lecture Setup — Redundant Column Problem

Dekho `drop_first=False` rakhne par har row me charo region columns me se exactly ek `1` hota hai aur baaki `0`.
- **Agla Concept (Lecture 2):** In charo columns ka row-sum hamesha `1` aana **Dummy Variable Trap** (multicollinearity) create karta hai.

In [ ]:
# 1. Region dummy columns ka row-wise sum check karo
cols = ["region_northeast", "region_northwest", "region_southeast", "region_southwest"]
row_sums = X[cols].sum(axis=1)

# 2. Verify karo ki har row ka sum strictly 1 hai
print("Unique row sums:", row_sums.unique())
print("Exact dependency verified — Lecture 2 me iska fix dekhenge!")